# 02. Preparación de Datos y Modelado de Aprendizaje Automático (ML)

## 1. Introducción y Objetivos de Preparación

En este cuaderno se implementan las transformaciones y reglas de limpieza documentadas durante la fase de exploración y auditoría de calidad (`01_exploracion_calidad.ipynb`). 

El propósito principal es consolidar el conjunto de datos masivo (~351 millones de escuchas) y estructurar las **Feature Matrices** específicas que alimentarán las técnicas de Aprendizaje Automático (No Supervisado y Supervisado):

1. **Aplicación de Reglas de Calidad:**
   * **Filtrado Temporal:** Eliminación de escuchas con marcas de tiempo anómalas (año 1970).
   * **Detección y Exclusión de Bots:** Filtrado de jornadas con `escuchas_dia > 720` reproducciones (actividad ininterrumpida anómala de 24 hs).
   * **Normalización Geográfica:** Remapeo explícito del código de país `UK` $\rightarrow$ `GB` (estándar ISO-2) para la integración limpia de datos.
   * **Tratamiento Demográfico:** Exclusión de usuarios con edades fuera del rango válido ($10 \le \text{age} \le 90$).

2. **Generación de Datasets Limpios (`/data/clean/`):**
   * **`df_paises_clean.parquet`:** Matriz agregada a nivel País (rasgos de audio de Spotify + Indulgencia/Individualismo de Hofstede + Felicidad de WHR). Entrada principal para la tarea de **Clustering**.
   * **`user_artist_matrix.npz`:** Matriz dispersa Usuario $\times$ Artista ponderada por frecuencia de escuchas para el **Sistema de Recomendación**.

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd
import numpy as np

# Configuración de rutas del proyecto
BASE_DIR = Path.cwd()
PARQUET_DIR = BASE_DIR / "data" / "parquet"
CLEAN_DIR = BASE_DIR / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Conexión a DuckDB
con = duckdb.connect()

print("--> Generando dataset limpio a nivel País para Clustering...")

query_paises = f"""
WITH bots AS (
    -- 1. Identificar jornadas de bots (>720 escuchas en un mismo día)
    SELECT 
        user_id, 
        strftime(to_timestamp(timestamp), '%Y-%m-%d') AS fecha
    FROM '{PARQUET_DIR}/events.parquet'
    GROUP BY user_id, fecha
    HAVING COUNT(*) > 720
),
escuchas_validas AS (
    -- 2. Filtrar escuchas entre 2005 y 2014, normalizar UK -> GB y excluir bots
    SELECT 
        e.user_id,
        CASE WHEN u.country = 'UK' THEN 'GB' ELSE u.country END AS country,
        af.valence, 
        af.energy, 
        af.danceability, 
        af.acousticness, 
        af.tempo
    FROM '{PARQUET_DIR}/events.parquet' e
    JOIN '{PARQUET_DIR}/users.parquet' u ON e.user_id = u.user_id
    JOIN '{PARQUET_DIR}/acoustic_features.parquet' af ON e.track_id = af.track_id
    LEFT JOIN bots b 
           ON e.user_id = b.user_id 
          AND strftime(to_timestamp(e.timestamp), '%Y-%m-%d') = b.fecha
    WHERE u.country IS NOT NULL AND u.country != ''
      AND year(to_timestamp(e.timestamp)) BETWEEN 2005 AND 2014
      AND b.user_id IS NULL  -- Exclusión explícita de registros de bots
),
metricas_sonoras AS (
    -- 3. Agregación a nivel país
    SELECT 
        country,
        COUNT(DISTINCT user_id) AS total_usuarios,
        COUNT(*) AS total_escuchas,
        AVG(valence) AS avg_valence,
        AVG(energy) AS avg_energy,
        AVG(danceability) AS avg_danceability,
        AVG(acousticness) AS avg_acousticness,
        AVG(tempo) AS avg_tempo
    FROM escuchas_validas
    GROUP BY country
    HAVING COUNT(DISTINCT user_id) >= 10  -- Mínimo 10 usuarios por país
)
SELECT 
    m.*,
    TRY_CAST(h.indulgence AS DOUBLE) AS indulgence,
    TRY_CAST(h.individualism AS DOUBLE) AS individualism,
    w."Life Ladder" AS happiness_score
FROM metricas_sonoras m
LEFT JOIN '{PARQUET_DIR}/hofstede.parquet' h ON m.country = h.ctr
LEFT JOIN '{PARQUET_DIR}/happiness.parquet' w ON m.country = w.country
ORDER BY total_escuchas DESC;
"""

df_paises = con.execute(query_paises).df()
df_paises.to_parquet(CLEAN_DIR / "df_paises_clean.parquet", index=False)

print(f"✓ Proceso finalizado. Dataset de países guardado en '{CLEAN_DIR / 'df_paises_clean.parquet'}'")
print(f"✓ Dimensión de la matriz: {df_paises.shape[0]} países por {df_paises.shape[1]} columnas.")
df_paises.head()

--> Generando dataset limpio a nivel País para Clustering...


---

## 2. Módulo 1: Clustering y Segmentación Cultural-Sonora (No Supervisado)

### 2.1 Planteo de la Tarea y Enfoque de ML
* **Objetivo de Aprendizaje Automático:** Descubrir patrones latentes y agrupar los países en "tribus sonoro-culturales" homogéneas evaluando si las preferencias de audio (alegría, energía, bailabilidad) se corresponden con las dimensiones culturales (Indulgencia, Individualismo) y el bienestar.
* **Técnicas a comparar:**
  1. **$K$-Means Clustering**
  2. **Clustering Jerárquico (Agglomerative)**
* **Variables seleccionadas ($X$):** `avg_valence`, `avg_energy`, `avg_danceability`, `avg_acousticness`, `avg_tempo`, `indulgence`, `individualism`, `happiness_score`.
* **Requisito de Cátedra (Hiperparámetros):** Evaluación de al menos 5 configuraciones distintas de $K$ ($K \in \{2, 3, 4, 5, 6, 7\}$) midiendo el impacto con Coeficiente Silhouette, Índice Davies-Bouldin e Inercia.

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd
import numpy as np

# Configuración de rutas del proyecto
BASE_DIR = Path.cwd()
PARQUET_DIR = BASE_DIR / "data" / "parquet"
CLEAN_DIR = BASE_DIR / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Conexión a DuckDB
con = duckdb.connect()

print("--> Generando dataset limpio a nivel País para Clustering...")

query_paises = f"""
WITH bots AS (
    -- 1. Identificar jornadas de bots (>720 escuchas en un mismo día)
    SELECT 
        user_id, 
        strftime(to_timestamp(timestamp), '%Y-%m-%d') AS fecha
    FROM '{PARQUET_DIR}/events.parquet'
    GROUP BY user_id, fecha
    HAVING COUNT(*) > 720
),
escuchas_validas AS (
    -- 2. Filtrar escuchas entre 2005 y 2014, normalizar UK -> GB y excluir bots
    SELECT 
        e.user_id,
        CASE WHEN u.country = 'UK' THEN 'GB' ELSE u.country END AS country,
        af.valence, 
        af.energy, 
        af.danceability, 
        af.acousticness, 
        af.tempo
    FROM '{PARQUET_DIR}/events.parquet' e
    JOIN '{PARQUET_DIR}/users.parquet' u ON e.user_id = u.user_id
    JOIN '{PARQUET_DIR}/acoustic_features.parquet' af ON e.track_id = af.track_id
    LEFT JOIN bots b 
           ON e.user_id = b.user_id 
          AND strftime(to_timestamp(e.timestamp), '%Y-%m-%d') = b.fecha
    WHERE u.country IS NOT NULL AND u.country != ''
      AND year(to_timestamp(e.timestamp)) BETWEEN 2005 AND 2014
      AND b.user_id IS NULL  -- Exclusión explícita de registros de bots
),
metricas_sonoras AS (
    -- 3. Agregación a nivel país
    SELECT 
        country,
        COUNT(DISTINCT user_id) AS total_usuarios,
        COUNT(*) AS total_escuchas,
        AVG(valence) AS avg_valence,
        AVG(energy) AS avg_energy,
        AVG(danceability) AS avg_danceability,
        AVG(acousticness) AS avg_acousticness,
        AVG(tempo) AS avg_tempo
    FROM escuchas_validas
    GROUP BY country
    HAVING COUNT(DISTINCT user_id) >= 10  -- Mínimo 10 usuarios por país
)
SELECT 
    m.*,
    TRY_CAST(h.indulgence AS DOUBLE) AS indulgence,
    TRY_CAST(h.individualism AS DOUBLE) AS individualism,
    w."Life Ladder" AS happiness_score
FROM metricas_sonoras m
LEFT JOIN '{PARQUET_DIR}/hofstede.parquet' h ON m.country = h.ctr
LEFT JOIN '{PARQUET_DIR}/happiness.parquet' w ON m.country = w.country
ORDER BY total_escuchas DESC;
"""

df_paises = con.execute(query_paises).df()
df_paises.to_parquet(CLEAN_DIR / "df_paises_clean.parquet", index=False)

print(f"✓ Proceso finalizado. Dataset de países guardado en '{CLEAN_DIR / 'df_paises_clean.parquet'}'")
print(f"✓ Dimensión de la matriz: {df_paises.shape[0]} países por {df_paises.shape[1]} columnas.")
df_paises.head()

--> Generando dataset limpio a nivel País para Clustering...
